# 03 - Stage 1 Pre-training (Colab)
Run masked generative pre-training for AudioCite.
Mount Google Drive to persist checkpoints across sessions.

In [6]:
# ── Colab only ─────────────────────────────────────────────────────────
import os
IN_COLAB = 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/audiocite'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    # Clone repo if not present
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/mohamedzait20003/ECE595NLP-Project /content/repo
    %cd /content/repo

    # Install dependencies
    !pip install -q -r requirements.txt
    !pip install -q torch --index-url https://download.pytorch.org/whl/cu124

    # Copy data from Drive to local (faster I/O during training)
    !cp -r {DRIVE_DIR}/data src/ 2>/dev/null || echo 'No cached data found in Drive'

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/repo') if IN_COLAB else Path().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 1. Configure Training

In [ ]:
import yaml

config_path = PROJECT_ROOT / 'src' / 'config' / 'pretrain_config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

if IN_COLAB:
    config['training']['checkpoint_dir'] = f'{DRIVE_DIR}/checkpoints/pretrain'

# Save modified config
with open(config_path, 'w') as f:
    yaml.dump(config, f)

print('Config:')
print(f"  total_steps : {config['training']['total_steps']}")
print(f"  batch_size  : {config['training']['batch_size']}")
print(f"  fp16        : {config['training']['fp16']}")
print(f"  checkpoint  : {config['training']['checkpoint_dir']}")

## 2. Run Training

In [ ]:
from src.main.training import train
train(str(config_path))

## 3. Plot Training Loss

In [ ]:
import json
import matplotlib.pyplot as plt

log_path = Path(config['training']['checkpoint_dir']) / 'training_log.json'

with open(log_path, 'r') as f:
    history = json.load(f)

steps = [h['step'] for h in history]
losses = [h['train_loss'] for h in history]

plt.figure(figsize=(12, 4))
plt.plot(steps, losses, linewidth=1)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Stage 1 Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {losses[-1]:.4f}')

## 4. Quick Sanity Check on Checkpoint

In [ ]:
import torch
from src.main.model.main_model import MainModel
from transformers import BartTokenizer

ckpt_path = Path(config['training']['checkpoint_dir']) / 'checkpoint_best.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')
print(f'Best checkpoint at step {ckpt["step"]} | val_loss: {ckpt["val_loss"]:.4f}')

# Load model and generate a sample
model = MainModel(whispher_model='openai/whisper-small', bart_model='facebook/bart-base')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# Dummy audio + context
dummy_audio = torch.randn(1, 80, 3000)
ctx = 'A Survey on Natural Language Processing with Deep Learning'
enc = tokenizer(ctx, return_tensors='pt', max_length=128, truncation=True)

with torch.no_grad():
    out = model.generate(
        dummy_audio,
        enc['input_ids'],
        enc['attention_mask'],
        max_length=32, num_beams=3
    )

print('Generated:', tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Save checkpoint back to Drive (Colab)
if IN_COLAB:
    import shutil
    shutil.copytree(
        config['training']['checkpoint_dir'],
        f'{DRIVE_DIR}/checkpoints/pretrain',
        dirs_exist_ok=True
    )
    print('Checkpoints saved to Google Drive.')